In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt

# --- CONFIGURATION & PARAMETERS ---
PHASES = ['North-South Straight', 'East-West Straight', 'North-South Left', 'East-West Left']
NUM_PHASES = len(PHASES)
MIN_GREEN, MAX_GREEN, YELLOW_TIME = 15, 60, 4

TRAFFIC_DEMAND = [0.45, 0.35, 0.15, 0.20]  # veh/s arriving
ROAD_CAPACITY = [0.90, 0.85, 0.50, 0.60]   # max discharge rate

POP_SIZE, GENERATIONS = 40, 50
CROSSOVER_RATE, MUTATION_RATE = 0.8, 0.2

# --- TRAFFIC SIMULATION & FITNESS ---
def evaluate_fitness(chromosome):
    cycle_length = sum(chromosome) + (NUM_PHASES * YELLOW_TIME)
    total_delay, total_vehicles = 0, 0
    max_queues = [0.0] * NUM_PHASES
    current_queues = [0.0] * NUM_PHASES

    # Generate continuous signal timeline
    timeline = []
    for phase_idx, green_time in enumerate(chromosome):
        timeline.extend([phase_idx] * green_time + [-1] * YELLOW_TIME)

    # Second-by-second micro-simulation
    for second in range(cycle_length):
        active_phase = timeline[second]
        for i in range(NUM_PHASES):
            current_queues[i] += TRAFFIC_DEMAND[i]
            if i == active_phase and current_queues[i] > 0:
                discharge = min(ROAD_CAPACITY[i], current_queues[i])
                current_queues[i] -= discharge
                total_vehicles += discharge
            max_queues[i] = max(max_queues[i], current_queues[i])
        total_delay += sum(current_queues)

    avg_delay = total_delay / total_vehicles if total_vehicles > 0 else 999.0
    fitness = 1000.0 / (avg_delay + 1.0)
    return fitness, avg_delay, max_queues

# --- EVOLUTIONARY ENGINE ---
population = [[random.randint(MIN_GREEN, MAX_GREEN) for _ in range(NUM_PHASES)] for _ in range(POP_SIZE)]
history_best, history_avg = [], []

print("Running Genetic Algorithm Optimization...")
for gen in range(GENERATIONS):
    evals = [evaluate_fitness(chrom) for chrom in population]
    fitness_scores = [ev[0] for ev in evals]
    delays = [ev[1] for ev in evals]

    # Track performance metrics
    best_idx = np.argmin(delays)
    history_best.append(delays[best_idx])
    history_avg.append(np.mean(delays))

    best_chrom = population[best_idx]
    best_queues = evals[best_idx][2]

    # Breed Next Generation (with Elitism)
    next_population = [best_chrom.copy()]
    while len(next_population) < POP_SIZE:
        # Tournament Selection
        p1 = max(random.sample(list(zip(population, fitness_scores)), 4), key=lambda x: x[1])[0]
        p2 = max(random.sample(list(zip(population, fitness_scores)), 4), key=lambda x: x[1])[0]

        # Single-Point Crossover
        if random.random() < CROSSOVER_RATE:
            pt = random.randint(1, NUM_PHASES - 1)
            c1, c2 = p1[:pt] + p2[pt:], p2[:pt] + p1[pt:]
        else:
            c1, c2 = p1.copy(), p2.copy()

        # Constrained Creep Mutation
        for child in [c1, c2]:
            if random.random() < MUTATION_RATE:
                idx = random.randint(0, NUM_PHASES - 1)
                child[idx] += random.choice([-7, -5, -3, 3, 5, 7])
                child[idx] = max(MIN_GREEN, min(child[idx], MAX_GREEN))
            if len(next_population) < POP_SIZE:
                next_population.append(child)

    population = next_population

# --- METRIC RELEASES ---
print(f"\nOptimization Complete!")
print(f"Initial Best Delay: {history_best[0]:.2f}s | Evolved Optimal Delay: {history_best[-1]:.2f}s")
print(f"Total Optimal Cycle Time: {sum(best_chrom) + (NUM_PHASES * YELLOW_TIME)} seconds")
for i, green in enumerate(best_chrom):
    print(f" - {PHASES[i]}: {green}s Green Allocation (Max Queue: {best_queues[i]:.1f} vehicles)")




Running Genetic Algorithm Optimization...

Optimization Complete!
Initial Best Delay: 62.57s | Evolved Optimal Delay: 55.71s
Total Optimal Cycle Time: 76 seconds
 - North-South Straight: 15s Green Allocation (Max Queue: 27.4 vehicles)
 - East-West Straight: 15s Green Allocation (Max Queue: 14.7 vehicles)
 - North-South Left: 15s Green Allocation (Max Queue: 5.7 vehicles)
 - East-West Left: 15s Green Allocation (Max Queue: 11.4 vehicles)
